In [2]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor

In [24]:
from sklearn.decomposition import PCA

In [ ]:
train_df = pd.read_csv(
    "/content/train.csv"
)

train_df.shape

In [ ]:
sample_df = train_df.sample(
    5000,
    random_state=42
).reset_index(drop=True)

sample_df.shape

In [6]:
X_train_text, X_valid_text, y_train, y_valid = train_test_split(
    sample_df["catalog_content"],
    sample_df["price"],
    test_size=0.2,
    random_state=42
)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_valid_tfidf = tfidf.transform(
    X_valid_text
)

print(X_train_tfidf.shape)
print(X_valid_tfidf.shape)

In [ ]:
def extract_quantity_features(text):
    text = str(text).lower()

    ounce = re.search(r'(\d+\.?\d*)\s*(oz|ounce)', text)
    pound = re.search(r'(\d+\.?\d*)\s*(lb|pound)', text)
    pack = re.search(r'pack of (\d+)', text)
    serving = re.search(r'(\d+)\s*servings', text)
    count = re.search(r'(\d+)\s*count', text)

    return pd.Series([
        float(ounce.group(1)) if ounce else 0,
        float(pound.group(1)) if pound else 0,
        float(pack.group(1)) if pack else 0,
        float(serving.group(1)) if serving else 0,
        float(count.group(1)) if count else 0
    ])


quantity_features = sample_df[
    "catalog_content"
].apply(extract_quantity_features)

quantity_features.columns = [
    "ounce_feature",
    "pound_feature",
    "pack_feature",
    "serving_feature",
    "count_feature"
]

quantity_features.head()

In [ ]:
quantity_train = quantity_features.loc[
    X_train_text.index
]

quantity_valid = quantity_features.loc[
    X_valid_text.index
]

print(quantity_train.shape)
print(quantity_valid.shape)

In [10]:
quantity_train_sparse = csr_matrix(
    quantity_train.values
)

quantity_valid_sparse = csr_matrix(
    quantity_valid.values
)

In [11]:
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import torch
import timm

from torchvision import transforms

In [ ]:
model = timm.create_model(
    "resnet50",
    pretrained=True,
    num_classes=0
)

model.eval()

In [13]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [14]:
def get_image_embedding(url):
    try:
        response = requests.get(
            url,
            timeout=10
        )

        image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")

        image = transform(image)

        image = image.unsqueeze(0)

        with torch.no_grad():
            embedding = model(image)

        return embedding.squeeze().numpy()

    except:
        return np.zeros(2048)

In [15]:
X_train_images = sample_df.loc[
    X_train_text.index,
    "image_link"
]

X_valid_images = sample_df.loc[
    X_valid_text.index,
    "image_link"
]

In [ ]:
train_embeddings = []

for url in tqdm(X_train_images):
    train_embeddings.append(
        get_image_embedding(url)
    )

train_embeddings = np.array(
    train_embeddings
)

In [17]:
valid_embeddings = []

for url in tqdm(X_valid_images):
    valid_embeddings.append(
        get_image_embedding(url)
    )

valid_embeddings = np.array(
    valid_embeddings
)

100%|██████████| 1000/1000 [04:10<00:00,  3.99it/s]


In [ ]:
print(train_embeddings.shape)
print(valid_embeddings.shape)

In [25]:
pca = PCA(n_components=100)

train_embeddings_pca = pca.fit_transform(
    train_embeddings
)

valid_embeddings_pca = pca.transform(
    valid_embeddings
)

print(train_embeddings_pca.shape)
print(valid_embeddings_pca.shape)

train_image_embeddings = csr_matrix(
    train_embeddings_pca
)

valid_image_embeddings = csr_matrix(
    valid_embeddings_pca
)

(4000, 100)
(1000, 100)


In [26]:
X_train_final = hstack([
    X_train_tfidf,
    quantity_train_sparse,
    train_image_embeddings
])

X_valid_final = hstack([
    X_valid_tfidf,
    quantity_valid_sparse,
    valid_image_embeddings
])

print(X_train_final.shape)
print(X_valid_final.shape)

(4000, 5105)
(1000, 5105)


In [ ]:
multimodal_model = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=50,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

multimodal_model.fit(
    X_train_final,
    y_train
)

In [ ]:
multimodal_predictions = multimodal_model.predict(
    X_valid_final
)

In [ ]:
mae = mean_absolute_error(
    y_valid,
    multimodal_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        multimodal_predictions
    )
)

r2 = r2_score(
    y_valid,
    multimodal_predictions
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)